# Model Development

This notebook documents feature engineering, model selection, and hyperparameter tuning for heart disease risk prediction. It calls straight
into the tested project modules (`heart_disease.features`, `heart_disease.train`) rather than re-implementing the logic, so what you see here is exactly what the CI pipeline and the served model run.

See `notebooks/eda.ipynb` for the exploratory analysis that motivated these choices.

In [ ]:
import pandas as pd
from IPython.display import Image, display
from sklearn.model_selection import train_test_split

from heart_disease.data import TARGET, get_clean_dataset
from heart_disease.features import build_pipeline, build_preprocessor
from heart_disease import train as train_module

pd.set_option("display.max_columns", 20)

## 1. Load the cleaned dataset and split

In [ ]:
df = get_clean_dataset()
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=train_module.RANDOM_STATE
)
print(f"train: {X_train.shape}, test: {X_test.shape}")
y_train.value_counts(normalize=True)

## 2. Feature engineering pipeline

`build_preprocessor()` (`src/heart_disease/features.py`) applies:
- `StandardScaler` to the 5 numeric features (`age`, `trestbps`, `chol`, `thalach`, `oldpeak`)
- `OneHotEncoder(handle_unknown="ignore", drop="if_binary")` to the 8 categorical features
  (`sex`, `cp`, `fbs`, `restecg`, `exang`, `slope`, `ca`, `thal`)

Wrapping this in one `sklearn.Pipeline` alongside the classifier guarantees training and
inference apply identical transformations there is no separate preprocessing script to
drift out of sync with the API.

In [ ]:
preprocessor = build_preprocessor()
transformed = preprocessor.fit_transform(X_train)
print("encoded feature matrix shape:", transformed.shape)
print("output feature names:", list(preprocessor.get_feature_names_out()))

## 3. Model selection & hyperparameter search space

Two classifiers are compared, each tuned with `GridSearchCV` (5-fold `StratifiedKFold`,
scored on ROC-AUC) — the search space is defined once in `heart_disease.train` and reused
here so the notebook and the training script can never disagree:

In [ ]:
for name, spec in train_module.MODEL_SEARCH_SPACE.items():
    print(name, "->", spec["param_grid"])

## 4. Train, tune, and track both models

`train_and_track` fits each model's pipeline with `GridSearchCV`, re-validates the best
estimator with cross-validation, evaluates it on the held-out test set, logs everything
(params, metrics, confusion matrix, ROC curve, model artifact) to MLflow, and persists the
pipelines under `models/`. Re-running this cell reproduces the full experiment from scratch.

In [ ]:
run_result = train_module.train_and_track(df)
print("best model:", run_result["best_model_name"])

## 5. Model comparison

In [ ]:
comparison = pd.DataFrame(
    {
        name: {**res["metrics"], "cv_roc_auc_mean": res["cv_roc_auc_mean"]}
        for name, res in run_result["results"].items()
    }
).T
comparison.sort_values("roc_auc", ascending=False)

## 6. Confusion matrices & ROC curves

Saved to `reports/figures/` by the training run above.

In [ ]:
for name in run_result["results"]:
    display(Image(filename=f"../reports/figures/confusion_matrix_{name}.png"))
    display(Image(filename=f"../reports/figures/roc_curve_{name}.png"))

## 7. Reproducibility check

Load the persisted best pipeline exactly the way the API does (`heart_disease.predict.
HeartDiseaseModel`) and confirm it reproduces a prediction on a held-out test row.

In [ ]:
from heart_disease.predict import FEATURE_ORDER, HeartDiseaseModel

model = HeartDiseaseModel()  # loads models/best_model.joblib
sample = {col: X_test.iloc[0][col] for col in FEATURE_ORDER}
print("input:", sample)
print("prediction:", model.predict_one(sample))
print("actual label:", int(y_test.iloc[0]))

## 8. Summary

- Logistic Regression and Random Forest were both tuned with `GridSearchCV` on ROC-AUC.
- Logistic Regression was selected as the best model on the held-out test set (highest
  ROC-AUC, consistent with cross-validation), and is what `models/best_model.joblib` and the
  FastAPI service (`api/main.py`) serve.
- Every run's parameters, metrics, and artifacts are tracked in MLflow — see
  `mlflow ui --backend-store-uri ../mlruns` (or `./mlruns` from the repo root).